[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PerformanceEstimation/Tutorial-SIAM-OP26/blob/main/Part%201%20[AM]%20-%20Semidefinite%20formulations%20and%20interpolation/SIAM_OP26_PEP_Minitutorial_part-1-hands-on.ipynb)

# First hands-on session with PEPit : numerical examples of worst-case analyses

PEPit is a package enabling **computer-assisted worst-case analyses** of first-order optimization methods. The key underlying idea is to cast the problem of performing a worst-case analysis, often referred to as a performance estimation problem (PEP), as a semidefinite program (SDP) which can be solved numerically. For doing that, the package users are only required to write first-order methods nearly as they would have implemented them. The package then takes care of the SDP modelling parts, and the worst-case analysis is performed numerically via a standard solver.


PEPit can be installed following these [instructions](https://pypi.org/project/PEPit/), and a quickstart is available in its [documentation](https://pepit.readthedocs.io/en/latest/). More information can be found in the [PEPit reference paper](https://arxiv.org/pdf/2201.04040.pdf).

We refer to [this blog post](https://francisbach.com/computer-aided-analyses/) and the references therein for a more mathematical introduction to performance estimation problems.

## 1 Installing PEPit

In [ ]:
# If PEPit is not installed yet, you can run this cell.
!pip3 install pepit;

## 2 How to obtain a worst-case guarantee for Gradient Descent using PEPit

In this section, we provide a step-by-step tutorial on how to use PEPit to compute a worst-case guarantee for gradient descent.

That is, we consider the minimization problem

$$\min_{x} f(x),$$

where $f$ is an $L$-smooth convex function which is minimized at $x_*$. We denote by $x_n$ the output of gradient descent with step-size $\gamma$ after $n\in\mathbb{N}$ iterations, starting from $x_0$. We perform a worst-case analysis in the following sense: we compute the smallest possible value of $\tau(L,\gamma,n)$ such that

$$ f(x_{n})-f(x_*) \leqslant\  \tau(L,\gamma,n)\  \  \|x_0-x_*\|^2_2$$

is valid for all $L$-smooth convex function $f$ and initial iterate $x_0$. Computing the smallest possible such $\tau(L,\gamma,n)$ is equivalent to computing the worst-case value of $f(x_{n})-f(x_*)$ under the constraint $\| x_{0}-x_*\|_2^2\leqslant 1$.

PEPit formulates the problem of computing $\tau(L,\gamma,n)$ as an SDP which can be solved numerically. In what follows, we describe the few lines that a user has to provide for generating the numerical worst-case analysis of GD using PEPit.

In general, performing a worst-case analysis of a first-order method usually relies on **four main ingredients**:
- <font color='blue'>a class of problems</font> (containing the assumptions on the function to be minimized),
- <font color='red'>a first-order method</font> (to be analyzed),
- <font color='purple'>a performance measure</font> (measuring the quality of the output of the algorithm under consideration),
- <font color='green'>an initial condition</font> (measuring the quality of the initial iterate).

PEPit requires the user to specify a choice for each of those four elements.

### 2.1 Imports
Before anything else, we have to import the PEP and the classes of functions of interest.

Note: for smooth convex functions we use smooth strongly convex functions with $\mu=0$

In [ ]:
from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction

### 2.2 Initialization of PEPit
Then, we initialize a PEP object. This object allows manipulating the forthcoming ingredients of the PEP, such as functions and iterates.

In [ ]:
# Instantiate PEP
problem = PEP()

### 2.3 Choose parameters values

For the sake of the example, we pick some simple values for the problem class and algorithmic parameters, for which we perform the worst-case analysis below.

In [ ]:
# Specify values for the smoothness parameter L, number of iterations n, and the step size gamma.
L = 1
n = 4
gamma = 1 / L

### 2.4 Specifying the problem class
Next, we specify our assumptions on the <font color='blue'>**class of functions**</font> containing the function to be optimized, and instantiate a corresponding object. Here, we consider an $L$-smooth convex function.

<br> <font color='grey'>*Remark*: PEPit can handle various function classes</font> [(documentation)](https://pepit.readthedocs.io/en/latest/api/functions_and_operators.html).

Note: we use the ```SmoothStronglyConvexFunction``` class with $\mu=0$ here. Alternatively, one could use ```SmoothConvexFunction```.

In [ ]:
# Declare an L-smooth mu-strongly convex function named "func"
func = problem.declare_function(
                SmoothStronglyConvexFunction,
                mu=0,   # Strong convexity param.
                L=L)    # Smoothness param.

We also define $x_*$ as an optimal point and $f_*$ as the optimal value, as those are needed to express our performance criteria and initial condition.

In [ ]:
# Start by defining an optimal point xs = x_* and corresponding function value fs = f_*
xs = func.stationary_point()
fs = func(xs)

### 2.5 Algorithm initialization

Third, we can instantiate the starting points for the two gradient methods that we will run, and specify an <font color='green'>**initial condition**</font> on those points.
To this end, a starting point $x_0$ is introduced,  and a bound on the initial distance between those points is specified as $\|x_0-x_*\|^2\leqslant 1$.

<br> <font color='grey'> *Remark*: PEPit can handle various initial constraints, for example any linear combination of $f(x_0)-f_*$, $\|x_0-x_*\|^2$ or $\|\nabla f(x_0)\|^2$.</font>


In [ ]:
# Then define the starting point x0 of the algorithm
x0 = problem.set_initial_point()

# Set the initial constraint that is the distance between x0 and x^*
problem.set_initial_condition((x0 - xs) ** 2 <= 1)

### 2.6 Algorithm implementation

In this fourth step, we specify the <font color='red'>**algorithm**</font> in a natural format. In this example, we simply use the iterates (which are PEPit objects) as if we had to implement gradient descent in practice using a simple loop. Gradient descent with fixed step size $\gamma$ may be described as follows, for $t \in \{0,1, \ldots, n-1\}$
\begin{equation}
x_{t+1} = x_t - \gamma \nabla f(x_t).
\end{equation}

<br> <font color='grey'>*Remark*: PEPit can handle most first order algorithms that rely on some simple</font> [oracles.](https://pepit.readthedocs.io/en/latest/api/steps.html)<br><font color='grey'> More than 90 examples (for deterministic, stochastic, and/or composite optimization) are provided in the</font> [documentation](https://pepit.readthedocs.io/en/latest/examples.html).

In [ ]:
# Run n steps of gradient descent with step-size gamma
x = x0
for _ in range(n):
    x = x - gamma * func.gradient(x)

### 2.7 Setting up a performance measure

It is crucial for the worst-case analysis to specify the <font color='purple'>**performance metric**</font> for which we wish to compute a worst-case performance. In this example, we wish to compute the worst-case value of $f(x_n)-f_*$, which we specify as follows.

<br> <font color='grey'> *Remark*: PEPit can handle various performance metrics, for example any linear combination of $f(x_n)-f_*$, $\|x_n-x_*\|^2$ or $\|\nabla f(x_n)\|^2$.</font>


In [ ]:
# Set the performance metric to the function values accuracy
problem.set_performance_metric(func(x) - fs)

### 2.8 Solving the PEP

The last natural stage in the process is to solve the corresponding PEP. This is done via the following line, which will ask PEPit to perform the modelling steps and to call an appropriate SDP solver (which should be installed beforehand) to perform the worst-case analysis.

Solving the SDP is handled by the [CVXPY](https://www.cvxpy.org/) modelling layer.

The default CVXPY solver [SCS](https://web.stanford.edu/~boyd/papers/scs.html)  may not be the most appropriate  for solving SDPs; we advise using [CLARABEL](https://clarabel.org/) (free, open-source, available in colab) or [MOSEK](https://www.mosek.com/) (commercial, free academic license) if possible, for improved numerical performances).

In [ ]:
# Import the CVXPY namespace for solver selection
import cvxpy as cp
# Solve the PEP with verbose = 1 and CLARABEL solver
pepit_tau = problem.solve(verbose=1, solver=cp.CLARABEL)
print(f"\n\n*** Exact worst-case convergence bound of GD for {L}-smooth convex function after {n} steps is {pepit_tau}")

### 2.10 Using the built-in PEPit function ````wc_gradient_descent````

It is possible to reproduce the above results  using the built-in ````wc_gradient_descent```` function, provided as on of the 90+ examples in the package, see  [`GradientMethod`](https://pepit.readthedocs.io/en/latest/examples/a.html#gradient-descent).

For the parameters specified above, the worst-case guarantee obtained numerically with the built-in PEPit function ````wc_gradient_descent```` matches the analytical value of the best known (tight) worst-case guarantee, which is also provided by the PEPit function.

That **analytical value** is provided in [1, Theorem 1]: when $\gamma \leqslant \frac{1}{L}$, the exact (tight) worst case guarantee is given by:
$$f(x_n)-f_* \leqslant \frac{L||x_0-x_\star||^2}{4nL\gamma+2}$$
which is achieved on some Huber loss functions.


[1] Y. Drori, M. Teboulle (2014). [Performance of first-order methods for smooth convex minimization: a novel approach](https://arxiv.org/pdf/1206.3209.pdf). Mathematical Programming 145(1–2), 451–482.


In [ ]:
from PEPit.examples.unconstrained_convex_minimization import wc_gradient_descent
pepit_tau, theoretical_tau = wc_gradient_descent(L=L, gamma=1 / L, n=4, verbose=1)

#### 2.11 Plotting worst-case guarantees as functions of number of steps

In [ ]:
import numpy as np
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize

# Set a list of numner of steps
ns = range(25)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
theoretical_taus = list()
for n in ns:
    pepit_tau, theoretical_tau = wc_gradient_descent(           L=L,
                                                                 gamma=gamma,
                                                                 n=n,
                                                                 verbose=0,
                                                                )
    pepit_taus.append(pepit_tau)
    theoretical_taus.append(theoretical_tau)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
import matplotlib.pyplot as plt
plt.plot(ns, theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps')
plt.ylabel('Worst-case guarantee')

plt.show()

inverse_pepit_taus = [1/tau for tau in pepit_taus]
inverse_theoretical_taus = [1/tau for tau in theoretical_taus]

plt.plot(ns, inverse_theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(ns, inverse_pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps')
plt.ylabel('Inverse worst-case guarantee')

plt.show()


#### 2.12 Plotting worst-case guarantees as functions of the step size

In [ ]:
import numpy as np
# Set the parameters
n = 1      # iteration counter
L = 1      # smoothness parameter

# Set a list of step-sizes to test
gammas = np.linspace(0, 2 / L, 41)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
theoretical_taus = list()
for gamma in gammas:
    pepit_tau, theoretical_tau = wc_gradient_descent(           L=L,
                                                                 gamma=gamma,
                                                                 n=n,
                                                                 verbose=0,
                                                                )
    pepit_taus.append(pepit_tau)
    theoretical_taus.append(theoretical_tau)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the step-size
import matplotlib.pyplot as plt
plt.plot(gammas, theoretical_taus, '--', label='Theoretical tight bound')
plt.plot(gammas, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Step-size')
plt.ylabel('Worst-case guarantee')

plt.show()

### Question: what is happening here with the theoretical bound?

(answer at the end of this morning's session)

#Now the real hands-on part start here: your turn!

## Task 1: compare between gradient descent, Nesterov acceleration, and an heavy-ball method

Consider the convex minimization problem

$$ \min_x f(x),$$

where $f$ is assumed to be convex and $L$-smooth. We compare three methods:

- vanilla gradient descent:

$$x_{k+1}=x_k-\tfrac1L\nabla f(x_k),$$

- an heavy-ball method:
\begin{equation*}
\begin{aligned}
x_{k+1} = x_k - \alpha \nabla f(x_k) + \beta (x_k - x_{k-1})
\end{aligned}
\end{equation*}
choice here: $\alpha = \tfrac{1}{2L}$, $\beta=\sqrt{1 - L\alpha}$, as studied in [1],

- a classical variation of Nesterov's accelerated gradient method [2]:
\begin{equation*}
\begin{aligned}
x_{k+1} &= y_k - \tfrac1L \nabla f(y_k)\\
y_{k+1} &= x_{k+1}+\tfrac{k}{k+3}(x_{k+1} - x_{k}).
\end{aligned}
\end{equation*}

---
**References.**

[1] E. Ghadimi, H.R. Feyzmahdavian and M. Johansson, *Global convergence of the Heavy-ball method for convex optimization*, European control conference (ECC), 2015. [arXiv:1412.7457](https://arxiv.org/abs/1412.7457)


[2] Y. Nesterov, *A method of solving a convex programming problem with convergence rate $O(1/k^2)$*, Doklady Akademii Nauk SSSR, 1983. [English translation](https://github.com/PerformanceEstimation/Tutorial-SMAI-MODE/blob/main/References/1983_Nesterov.pdf)

The performance criterion we will use $\frac{f(x_n)-f_\star}{\|x_0-x_\star\|}$. In other words, due to homogeneity of the worst-case rate function, we are going to compute the worst-case final objective accuracy $f(x_n)-f_\star$ given an initial condition on the starting point $\|x_0-x_\star\| \le 1$.

##3.1 Fill the missing parts in the three codes below, and test them on the suggested examples

Missing parts are denoted by ``###``. They consist, for each algorithm, in the definitions of the step and the  performance criteria.

### Gradient descent

In [ ]:
from PEPit.functions import SmoothConvexFunction
from math import sqrt

def wc_gradient_descent(L, n):

    gamma = 1/L
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth convex function
    f = problem.declare_function(SmoothConvexFunction, L=L)

    # Start by defining its unique optimal point xs = x_* and corresponding function value fs = f_*
    xs = f.stationary_point()
    fs = f(xs)

    # Then define the starting point x0 of the algorithm
    x0 = problem.set_initial_point()

    # Set the initial constraint that is the distance between x0 and x^*
    problem.set_initial_condition((x0 - xs) ** 2 <= 1)

    # Run n steps of the GD method
    x = x0
    for _ in range(n):
        x = x -  ### write the gradient step here

    # Set the performance metric to the function values accuracy
    problem.set_performance_metric( ### write the performance metric f(xn)-f* here )

    # Solve the PEP
    pepit_tau = problem.solve(verbose=0, solver=cp.CLARABEL)

    # Return the worst-case guarantee of the evaluated method
    return pepit_tau

### Check your code on these two examples

In [ ]:
wc_gradient_descent(1, 1)	# should return 0.166671

In [ ]:
wc_gradient_descent(1, 10)	# should return 0.023809

### Heavy ball with momentum

In [ ]:
def wc_heavy_ball_momentum(L, n):

    alpha = .5/L
    beta = sqrt((1 - L*alpha))
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth strongly convex function
    f = problem.declare_function(SmoothConvexFunction, L=L)

    # Start by defining its unique optimal point xs = x_* and corresponding function value fs = f_*
    xs = f.stationary_point()
    fs = f(xs)

    # Then define the starting point x0 of the algorithm as well as corresponding function value f0
    x0 = problem.set_initial_point()
    f0 = f(x0)

    # Set the initial constraint that is the distance between f(x0) and f(x^*)
    problem.set_initial_condition((x0-xs)**2 <= 1)

    # Run one step of the heavy ball method
    x_new = x0
    x_old = x0

    for _ in range(n):
        x_next = x_new - ### write the heavy ball step here
        x_old = x_new
        x_new = x_next

    # Set the performance metric to the function value accuracy on last iterate x_new
    problem.set_performance_metric( ### write the performance metric f(xn)-f* here )

    # Solve the PEP
    pepit_tau = problem.solve(verbose=0, solver=cp.CLARABEL)

    # Return the worst-case guarantee of the evaluated method
    return pepit_tau

### Check your code on these two examples

In [ ]:
wc_heavy_ball_momentum(1, 1)	# should return 0.250000

In [ ]:
wc_heavy_ball_momentum(1, 10)	# should return 0.059041

### Accelerated gradient descent

In [ ]:

def wc_accelerated_gradient_convex(L, n):

    # Instantiate PEP
    problem = PEP()

    # Declare a strongly convex smooth function
    f = problem.declare_function(SmoothConvexFunction, L=L)

    # Start by defining its unique optimal point xs = x_* and corresponding function value fs = f_*
    xs = f.stationary_point()
    fs = f(xs)

    # Then define the starting point x0 of the algorithm
    x0 = problem.set_initial_point()

    # Set the initial constraint that is the distance between x0 and x^*
    problem.set_initial_condition((x0 - xs) ** 2 <= 1)

    # Run n steps of the fast gradient method
    x_new = x0
    y = x0
    for i in range(n):
        x_old = x_new
        x_new = ### write heavy ball step here
        y = ### write heavy ball step here

    # Set the performance metric to the function value accuracy on last iterate y
    problem.set_performance_metric( ### write the performance metric f(yn)-f* here )

    # Solve the PEP
    pepit_tau = problem.solve(verbose=0, solver=cp.CLARABEL)

    # Return the worst-case guarantee of the evaluated method (and the reference theoretical value)
    return pepit_tau

### Check your code on these two examples

In [ ]:
wc_accelerated_gradient_convex(1, 1) 	# should return 0.166671

In [ ]:
wc_accelerated_gradient_convex(1, 10)	# should return 0.011494

## 3.2 Then execute your code below

In [ ]:
# Execute for different parameters (will be a bit slower, due to n=30 and n=40)
import matplotlib.pyplot as plt
import numpy as np
import time

L = 1
n_list = np.array([1, 2, 3, 4, 5, 6, 7, 8, 10, 15, 20, 25, 30, 40])

pepit_taus_GD = list()
pepit_taus_HB = list()
pepit_taus_AGD = list()

count = 0
for n in n_list:
    pepit_taus_GD.append(wc_gradient_descent(L, n))
    pepit_taus_HB.append(wc_heavy_ball_momentum(L, n))
    pepit_taus_AGD.append(wc_accelerated_gradient_convex(L, n))
    count += 1
    print(f'{count} / {len(n_list)} points computed', end='\r', flush=True)
print("done!                          ")


## 3.3 Then plot the results

In [ ]:
# Create figure and axes
fig, ax = plt.subplots()

# Plot the data

ax.plot(n_list, pepit_taus_HB, '--', label='Heavy-ball', linewidth=2)
ax.plot(n_list, pepit_taus_GD, '-', label='Gradient descent', linewidth=2)
ax.plot(n_list, pepit_taus_AGD, '--', label='Accelerated gradient', linewidth=2)

# Add legend, labels, and configure log-log scale
legend = ax.legend(fontsize=16)
legend.get_frame().set_facecolor('none')  # Make the legend box background transparent
legend.get_frame().set_edgecolor('none')  # Make the legend box border transparent

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_xlabel('Iteration count n', labelpad=5, color='#333333', fontsize = 18)
ax.set_ylabel(r'Worst-case $\frac{f(x_n)-f_\star}{\|x_0-x_\star\|^2}$', labelpad=5, color='#333333',fontsize = 18)

# Save the figure before displaying it
plt.savefig(fname='func_values.eps', format='eps')

# Show the plot
plt.show()

## Example 2 : gradient descent for smooth (potentially) non-convex minimization <a class="anchor" id="example3"></a>

We consider the following minimization problem:
\begin{equation}
f_\star \triangleq \min_x f(x),
\end{equation}
where $f$ is $L$-smooth, and not assumed to be convex (hence potentially non-convex).

For this example, we consider the problem of computing the smallest possible $\tau(n, L, \mu, \gamma)$ such that the following guarantee holds (for any initialization of the algorithm, and any $L$-smooth function)
\begin{equation}
\min_{0\leq t\leq n} \|\nabla f(x_t)\|^2 \leqslant \tau(n, L, \gamma) (f(x_0)-f(x_\star)),
\end{equation}
where $x_t$ are the iterates gradient descent with step size $\gamma$, started from $x_0$:
\begin{equation}
x_{t+1} = x_t - \gamma \nabla f(x_t),
\end{equation}
and where $x_\star$ is a stationary point such that $f(x_\star)\leq f(x_t)$ for all $t=0,1,\ldots,n$.


In [ ]:
from PEPit import PEP
from PEPit.functions import SmoothFunction

L = 1
n = 3
gamma = .95/L
verbose = 1


# Instantiate PEP
problem = PEP()

# Declare a smooth strongly convex function
f = problem.declare_function(SmoothFunction, L=L) #TODO OPERAND

# Then define the starting point x0 of the algorithm as well as corresponding gradient and function value g0 and f0
x0 = problem.set_initial_point()
g0, f0 = f.oracle(x0)

xs = f.stationary_point()
gs, fs = f.oracle(xs)

# Run n steps of GD method with step-size gamma
x_list = list()
g_list = list()
f_list = list()
gx, fx = g0, f0

# All x, f and g across iterations are accumulated in lists for later use
x_list.append(x0)
f_list.append(f0)
g_list.append(g0)

for i in range(n):
    # Set the performance metric to the minimum of the gradient norm over the iterations
    problem.set_performance_metric(gx**2)      # declaring several performance metric will optimize their minimum
    f.add_constraint(fs <= fx - 1/2/L * gx**2)
    # Define here the next step (to be added to the list)
    x_list.append(x_list[-1] - ### define step here)
    gx, fx = f.oracle(x_list[-1])
    f_list.append(fx)
    g_list.append(gx)
problem.set_performance_metric(gx**2)
f.add_constraint(fs <= fx - 1/2/L * gx**2)

# Set initial condition
problem.set_initial_condition( ### define initial condition here )

# Solve the PEP (see later for explanation of "logdet3" heuristic)
pepit_tau = problem.solve(verbose=1, dimension_reduction_heuristic="logdet1", tol_dimension_reduction=1e-6, solver=cp.CLARABEL)

#### Low-dimensional worst-case examples

The solution to the PEP provides an example trajectory of points that would result in a worst-case convergence rate. In general, worst-case examples are non-unique in different ways: there are generally many trajectories of points that result in worst-case performance, and there are generally multiple different functions that can interpolate the same trajectory. Using a solution to the PEP, one can identify the dimension of a matching worst-case instance by inspection of the rank of the Gram matrix. More precisely, the rank of the Gram matrix corresponds to the dimension of the matching worst-case example (see, e.g., [here, Section 3.2](https://arxiv.org/pdf/1502.05666)).

PEPit contains a few heuristics to try to identify such low-dimensional worst-case instances, by searching for low-rank Gram matrices. There are two of them: the trace heuristic ($\ell_1$ on the eigenvalues) and the logdet heuristic---applying gradient descent on the logdet of the Gram matrix $G$, which corresponds to a reweighted $\ell_1$ on the eigenvalues of $G$. For both of them, we need to potentially leave some slack in the objective, which we fix here to `tol_dimension_reduction=1e-6`.

As we can see from the output, PEPit identified a low-rank matrix where all eigenvalues (except one) is approximately equal to 0. This corresponds to a single-dimensional worst-case example.
In order to evaluate and plot a worst-case trajectory, we use the `eval` function on the iterates, their gradients, and function values. Since we want to visualize the worst-case in 1D, we trim the dimension to only the first components.

In [ ]:
# Evaluate the iterations:

x_list_evaluated = [x.eval()[0] for x in x_list] # [0] is because we are interested in largest component only
g_list_evaluated = [g.eval()[0] for g in g_list] # same
f_list_evaluated = [f.eval() for f in f_list]

# Evaluate a specific point (x*):
xs_evaluated = xs.eval()[0] # x*
fs_evaluated = fs.eval() # f(x*)
gs_evaluated = gs.eval()[0] # g(x*)

# Evaluate a specific point (x0):
x0_evaluated = xs.eval()[0] # x*

We provide a few tools that allows extrapolating within certain classes of functions. This is done via the "interpolator" classes (that internally formulate the interpolation problems for a few classes).

The interpolator that corresponds to the PEP under consideration can be obtained (when implemented for the corresponding class) as follows (possibly with a few options):

In [ ]:
feval = f.get_interpolator()

Now, we can evaluate an interpolating function on a grid of points "x_test" around the iterates and the solution.

In [ ]:
# Create a grid of points where an interpolating function for f() should be computed:
nb_pts_grid = 100 # number of points on the grid
margin = .2 # extrapolate outside only evaluated points

x_min = np.min([np.min(x_list_evaluated),xs_evaluated])
x_max = np.max([np.max(x_list_evaluated),xs_evaluated])
x_test = np.linspace(x_min-margin,x_max+margin,nb_pts_grid)

fx_test = np.zeros(x_test.shape)
# Interpolate on the grid!
for i in range(len(x_test)):
    fx_test[i] = feval(np.array([x_test[i]]))
    print(f'{i + 1} / {len(x_test)} grid points computed', end='\r', flush=True)


... which we can now plot!

In [ ]:
plt.plot(x_test, fx_test, '--', color='blue')
plt.plot(x_list_evaluated, f_list_evaluated, marker='.', color='red',  linestyle='none', markersize=8, label=r'$(x_t,f(x_t))$')
plt.plot(xs_evaluated, fs_evaluated, marker='*', color='gold', linestyle='none', markersize=8, label=r'$(x_\star,f(x_\star))$')


plt.legend()
plt.xlabel('$x$')
plt.ylabel('$f(x)$')

plt.show()

A natural question now is whether this interpolator is actually unique. In fact, it is not in general. In PEPit, the default interpolator is often the one that creates the *lowest possible* interpolating function (the one that results in points with the *smallest possible function value*), but one can configure the interpolator to do the opposite (as we shall see in the next example).

In [ ]:
# get an interpolator
feval = f.get_interpolator(options='highest')


# Create a grid of points where an interpolating function for f() should be computed:
nb_pts_grid = 100 # number of points on the grid
margin = .2 # extrapolate outside only evaluated points

x_min = np.min([np.min(x_list_evaluated),xs_evaluated])
x_max = np.max([np.max(x_list_evaluated),xs_evaluated])
x_test = np.linspace(x_min-margin,x_max+margin,nb_pts_grid)

fx_test = np.zeros(x_test.shape)
# Interpolate on the grid!
for i in range(len(x_test)):
    fx_test[i] = feval(np.array([x_test[i]]))
    print(f'{i + 1} / {len(x_test)} grid points computed', end='\r', flush=True)


In [ ]:
feval_lowest = f.get_interpolator()
fx_test_lowest = np.array([feval_lowest(np.array([x])) for x in x_test])

plt.plot(x_test, fx_test_lowest, '--', color='blue', label='Lowest interpolating function')
plt.plot(x_test, fx_test, '--', color='orange', label='Highest interpolating function')
plt.plot(x_list_evaluated, f_list_evaluated, marker='.', color='red',  linestyle='none', markersize=8, label='Shared interpolation points')
plt.plot(xs_evaluated, fs_evaluated, marker='*', color='gold', linestyle='none', markersize=8, label='Shared stationary point')


plt.legend()
plt.xlabel('$x$')
plt.ylabel('$f(x)$')

plt.show()

This is slightly different from the previous example (particularly outside of the range $[x_\star,x_0]$).